# TinyStories KV Cache Benchmark: MHA, MQA, GQA, and MLA-style Attention

This notebook is an educational benchmark about one inference-engineering question:

> When a language model is generating the next token, how much does the KV cache cost, and how do MQA, GQA, and MLA-style caching reduce that cost?

We will anchor the benchmark to **TinyStories** so the context tokens come from real text. We will not train a model here. Instead, we will simulate the inference-time cache shapes used by different attention mechanisms and time the long-context decode step.

By the end, you should understand:

- why MHA has the largest KV cache
- why MQA has the smallest KV cache
- why GQA is a practical compromise
- why MLA-style latent caching is a different compression idea
- why these differences matter more as context length grows


## 1. Setup

Run this notebook on a CUDA GPU. H100/H200/B200 is ideal, but you can reduce the context lengths if you are using a smaller GPU.


In [ ]:
# If needed on a fresh RunPod image:
# !pip install -q torch datasets tiktoken pandas matplotlib tqdm

import math
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
import tiktoken
from IPython.display import display

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU for CUDA-event timing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")

device = "cuda"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"dtype: {dtype}")

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True


## 2. TinyStories Contexts

During inference, the model first reads the prompt. This is called **prefill**. During prefill, the model creates a KV cache for all prompt tokens.

For this notebook, the prompt tokens come from TinyStories. We concatenate TinyStories examples into one long token stream and sample contiguous windows from it.

The file below is just a local cache:

```python
TOKEN_CACHE = Path("tinystories_token_stream.npy")
```

You do not need to download this file manually. The notebook creates it automatically the first time it runs.


In [ ]:
TOKEN_CACHE = Path("tinystories_token_stream.npy")
DATASET_NAME = "roneneldan/TinyStories"
EOS_ID = 50256

enc = tiktoken.get_encoding("gpt2")


def build_or_load_tinystories_stream(min_tokens: int) -> np.ndarray:
    if TOKEN_CACHE.exists():
        tokens = np.load(TOKEN_CACHE, mmap_mode="r")
        if len(tokens) >= min_tokens:
            print(f"Loaded cached TinyStories token stream: {len(tokens):,} tokens")
            return np.asarray(tokens[:min_tokens], dtype=np.int64)

    print(f"Streaming TinyStories until we have at least {min_tokens:,} tokens...")
    ds = load_dataset(DATASET_NAME, split="train", streaming=True)
    chunks = []
    total = 0
    for i, ex in enumerate(ds):
        ids = enc.encode_ordinary(ex["text"])
        ids.append(EOS_ID)
        arr = np.asarray(ids, dtype=np.uint16)
        chunks.append(arr)
        total += len(arr)
        if (i + 1) % 1000 == 0:
            print(f"  read {i + 1:,} stories, {total:,} tokens")
        if total >= min_tokens:
            break

    tokens = np.concatenate(chunks).astype(np.uint16)
    np.save(TOKEN_CACHE, tokens)
    print(f"Saved {TOKEN_CACHE} with {len(tokens):,} tokens")
    return tokens[:min_tokens].astype(np.int64)


def sample_context_batch(tokens: np.ndarray, batch: int, context: int, seed: int = 0):
    rng = np.random.default_rng(seed + batch * 1009 + context)
    max_start = len(tokens) - context - 2
    if max_start <= 0:
        raise ValueError(f"Need more TinyStories tokens for context={context:,}")
    starts = rng.integers(0, max_start, size=batch)
    x = np.stack([tokens[s : s + context] for s in starts])
    next_tok = np.asarray([tokens[s + context] for s in starts])
    return torch.tensor(x, device=device, dtype=torch.long), torch.tensor(next_tok, device=device, dtype=torch.long)


## 3. What Exactly Are We Timing?

For every new generated token, attention compares the new token's query vector against the cached keys from all previous tokens:

```text
scores = q @ K.T
probs  = softmax(scores)
out    = probs @ V
```

The expensive object is the **KV cache**:

- `K` stores key vectors for previous tokens
- `V` stores value vectors for previous tokens
- longer context means more cached tokens
- more KV heads means more cached vectors per token

This notebook times only the decode attention step after the cache already exists. That isolates the inference bottleneck we care about.


## 4. A Short RoPE Subsection

Modern LLMs usually need some way to know token positions. One common method is **RoPE**, short for Rotary Positional Embeddings.

For this notebook, you only need the intuition:

> RoPE rotates query and key vectors by an amount that depends on token position.

So token 10 and token 10,000 do not have identical positional meaning. In real decoder-only models, RoPE is applied to `q` and `k` before attention.

We include RoPE here because it makes the cache construction more realistic. But RoPE is not the main lesson. The main lesson is still KV-cache size.


In [ ]:
def rotate_half(x):
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def rope_cache(seq_len: int, dim: int, device: str, dtype: torch.dtype, base: float = 10000.0):
    if dim % 2 != 0:
        raise ValueError("RoPE dimension must be even")
    pos = torch.arange(seq_len, device=device, dtype=torch.float32)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, device=device, dtype=torch.float32) / dim))
    freqs = torch.einsum("t,d->td", pos, inv_freq)
    emb = torch.repeat_interleave(freqs, 2, dim=-1)
    return emb.cos().to(dtype), emb.sin().to(dtype)


def apply_rope(x, cos, sin):
    # x: [B, H, T, D]
    return (x * cos[None, None, :, :]) + (rotate_half(x) * sin[None, None, :, :])


## 5. Cache Builders

The classes below convert TinyStories token IDs into attention-shaped tensors.

They are intentionally lightweight. They are not full trained transformer blocks. Their job is to create realistic cache shapes so we can compare inference memory movement.


In [ ]:
class TokenKVBuilder(nn.Module):
    """Build RoPE-aware q/k/v caches from TinyStories token IDs."""

    def __init__(self, vocab_size: int, q_heads: int, kv_heads: int, head_dim: int):
        super().__init__()
        self.q_heads = q_heads
        self.kv_heads = kv_heads
        self.head_dim = head_dim
        self.q_embed = nn.Embedding(vocab_size, q_heads * head_dim)
        self.k_embed = nn.Embedding(vocab_size, kv_heads * head_dim)
        self.v_embed = nn.Embedding(vocab_size, kv_heads * head_dim)

    @torch.inference_mode()
    def build(self, context_tokens, next_tokens):
        B, S = context_tokens.shape
        q = self.q_embed(next_tokens).view(B, self.q_heads, 1, self.head_dim)
        k = self.k_embed(context_tokens).view(B, S, self.kv_heads, self.head_dim).transpose(1, 2).contiguous()
        v = self.v_embed(context_tokens).view(B, S, self.kv_heads, self.head_dim).transpose(1, 2).contiguous()

        cos, sin = rope_cache(S + 1, self.head_dim, device, dtype)
        k = apply_rope(k, cos[:S], sin[:S])
        q = apply_rope(q, cos[S : S + 1], sin[S : S + 1])
        return q.contiguous(), k.contiguous(), v.contiguous()


class TokenLatentBuilder(nn.Module):
    """Build an MLA-style compressed latent cache from TinyStories token IDs."""

    def __init__(self, vocab_size: int, q_heads: int, latent_dim: int):
        super().__init__()
        self.q_heads = q_heads
        self.latent_dim = latent_dim
        self.q_embed = nn.Embedding(vocab_size, q_heads * latent_dim)
        self.latent_embed = nn.Embedding(vocab_size, latent_dim)

    @torch.inference_mode()
    def build(self, context_tokens, next_tokens):
        B, S = context_tokens.shape
        q = self.q_embed(next_tokens).view(B, self.q_heads, 1, self.latent_dim)
        latent = self.latent_embed(context_tokens).contiguous()

        # Pedagogical approximation: give the latent cache positional structure too.
        cos, sin = rope_cache(S + 1, self.latent_dim, device, dtype)
        latent = (latent * cos[:S][None, :, :]) + (rotate_half(latent) * sin[:S][None, :, :])
        q = apply_rope(q, cos[S : S + 1], sin[S : S + 1])
        return q.contiguous(), latent.contiguous()


## 6. Timing Helpers

The code below records CUDA-event timings and computes the metrics we will compare.

The two most important columns are:

- `cache_mb`: how large the physical cache is
- `tokens_per_sec`: how many decode steps per second the benchmark achieves, multiplied by batch size


In [ ]:
@dataclass
class BenchRow:
    method: str
    batch: int
    context: int
    q_heads: int
    kv_heads: int
    head_dim: int
    cache_mb: float
    bytes_per_token_kb: float
    arithmetic_intensity: float
    ms_per_step: float
    tokens_per_sec: float
    approx_gb_s: float
    speedup_vs_mha: float | None = None
    cache_reduction_vs_mha: float | None = None


def cuda_sync():
    torch.cuda.synchronize()


def time_cuda(fn, warmup=10, iters=50):
    for _ in range(warmup):
        fn()
    cuda_sync()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn()
    end.record()
    cuda_sync()
    return start.elapsed_time(end) / iters


def kv_cache_bytes(batch, context, kv_heads, head_dim, element_size):
    return 2 * batch * context * kv_heads * head_dim * element_size


def attention_flops(batch, context, q_heads, head_dim):
    return 4 * batch * q_heads * context * head_dim


def bench_grouped_decode_from_cache(method, q, k, v, warmup, iters):
    B, q_heads, _, head_dim = q.shape
    _, kv_heads, context, _ = k.shape
    if q_heads % kv_heads != 0:
        raise ValueError(f"q_heads={q_heads} must be divisible by kv_heads={kv_heads}")

    rep = q_heads // kv_heads
    q_grouped = q.view(B, kv_heads, rep, 1, head_dim)
    scale = 1.0 / math.sqrt(head_dim)

    def step():
        scores = torch.einsum("bhrtd,bhsd->bhrts", q_grouped, k) * scale
        probs = torch.softmax(scores, dim=-1)
        return torch.einsum("bhrts,bhsd->bhrtd", probs, v)

    ms = time_cuda(step, warmup=warmup, iters=iters)
    elem = q.element_size()
    cache_bytes = kv_cache_bytes(B, context, kv_heads, head_dim, elem)
    flops = attention_flops(B, context, q_heads, head_dim)

    return BenchRow(
        method=method,
        batch=B,
        context=context,
        q_heads=q_heads,
        kv_heads=kv_heads,
        head_dim=head_dim,
        cache_mb=cache_bytes / (1024**2),
        bytes_per_token_kb=(cache_bytes / B) / 1024,
        arithmetic_intensity=flops / cache_bytes,
        ms_per_step=ms,
        tokens_per_sec=B * 1000.0 / ms,
        approx_gb_s=cache_bytes / (ms / 1000.0) / 1e9,
    )


def bench_mla_latent_from_cache(q, latent, warmup, iters):
    B, q_heads, _, latent_dim = q.shape
    context = latent.shape[1]
    scale = 1.0 / math.sqrt(latent_dim)

    def step():
        scores = torch.einsum("bhld,bsd->bhls", q, latent) * scale
        probs = torch.softmax(scores, dim=-1)
        return torch.einsum("bhls,bsd->bhld", probs, latent)

    ms = time_cuda(step, warmup=max(3, warmup // 2), iters=max(10, iters // 2))
    elem = q.element_size()
    cache_bytes = 2 * B * context * latent_dim * elem
    flops = 4 * B * q_heads * context * latent_dim

    return BenchRow(
        method="MLA-style latent",
        batch=B,
        context=context,
        q_heads=q_heads,
        kv_heads=1,
        head_dim=latent_dim,
        cache_mb=cache_bytes / (1024**2),
        bytes_per_token_kb=(cache_bytes / B) / 1024,
        arithmetic_intensity=flops / cache_bytes,
        ms_per_step=ms,
        tokens_per_sec=B * 1000.0 / ms,
        approx_gb_s=cache_bytes / (ms / 1000.0) / 1e9,
    )


## 7. Experimental Design: What Are We Varying?

Before running the benchmark, we need to define the experiment clearly.

We vary two workload dimensions:

| Variable | Meaning | Why it matters |
|---|---|---|
| `context` | How many previous TinyStories tokens are already in the prompt/cache | Longer context means a larger KV cache must be read for every generated token |
| `batch` | How many independent sequences are decoded together | Larger batch gives the GPU more parallel work and better exposes memory-bandwidth effects |

A single benchmark row means:

```text
method = one attention mechanism, such as MHA or MQA
batch = number of TinyStories sequences decoded at once
context = number of previous tokens per sequence
operation = one next-token decode attention step
```

For example:

```text
method = MHA
batch = 16
context = 32768
```

means: we sample 16 TinyStories contexts, each 32,768 tokens long, build the MHA-style KV cache, and time how fast one decode attention step runs.

The benchmark is not asking, "Can this model write a good story?" The benchmark is asking:

> Once the model already has a long TinyStories context in its cache, how expensive is the next-token attention step?


In [ ]:
BATCH_SIZES = [1, 4, 16]
CONTEXTS = [1024, 4096, 8192, 16384, 32768]
Q_HEADS = 32
HEAD_DIM = 128
GQA_KV_HEADS = 8
MLA_LATENT_DIM = 512
WARMUP = 10
ITERS = 50
VOCAB_SIZE = enc.n_vocab

min_tokens = max(CONTEXTS) * max(BATCH_SIZES) + 100_000
token_stream = build_or_load_tinystories_stream(min_tokens)
print(f"Benchmark token stream length: {len(token_stream):,}")

rows = []


## 8. How to Read the Experiments

For each attention mechanism, we run the same grid of experiments:

```text
batch sizes:     1, 4, 16
context lengths: 1K, 4K, 8K, 16K, 32K tokens
```

Why these values?

- `batch = 1` shows the single-user case, but it can be noisy because GPU launch overhead matters a lot.
- `batch = 4` is a middle ground where the GPU has more useful work.
- `batch = 16` is the most useful teaching view because the KV cache becomes large enough for memory bandwidth to dominate.
- `context = 1024` is short-context inference.
- `context = 32768` is long-context inference, where KV-cache design matters much more.

Each method section below runs the same experiment grid. That is what makes the final comparison fair.


## 9. What Metrics Will We Collect?

The results table contains several columns, but four are the most important:

| Metric | Meaning |
|---|---|
| `cache_mb` | Total physical cache size for that batch and context |
| `bytes_per_token_kb` | How many cache KB each sequence carries per generated token |
| `tokens_per_sec` | Decode throughput for this one attention step |
| `speedup_vs_mha` | How much faster this method is than MHA for the same batch/context |

The most important sanity check is the cache reduction ratio:

```text
MHA: baseline
GQA with 8 KV heads: 4x smaller cache than MHA
MQA with 1 KV head: 32x smaller cache than MHA
MLA-style latent with dim 512: 8x smaller cache than MHA in this setup
```

If the cache-size numbers do not follow this pattern, the experiment is wrong.



## 10. Let's Now Do MHA

**Multi-Head Attention (MHA)** is the baseline.

In MHA, every query head has its own key head and its own value head:

```text
32 query heads -> 32 key heads + 32 value heads
```

This gives every head maximum freedom, but it also creates the largest KV cache. For long-context inference, MHA is the most memory-hungry option.


In [ ]:
for batch in BATCH_SIZES:
    for context in CONTEXTS:
        context_tokens, next_tokens = sample_context_batch(token_stream, batch=batch, context=context, seed=42)
        try:
            builder = TokenKVBuilder(VOCAB_SIZE, Q_HEADS, Q_HEADS, HEAD_DIM).to(device=device, dtype=dtype).eval()
            with torch.inference_mode():
                q, k, v = builder.build(context_tokens, next_tokens)
            row = bench_grouped_decode_from_cache("MHA", q, k, v, WARMUP, ITERS)
            rows.append(row)
            print(f"MHA batch={batch:2d} context={context:5d} cache={row.cache_mb:8.1f} MB tok/s={row.tokens_per_sec:10.1f}")
            del builder, q, k, v
            torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"OOM: MHA batch={batch}, context={context}")


## 11. Let's Now Do MQA

**Multi-Query Attention (MQA)** keeps all query heads but shares one key head and one value head across them:

```text
32 query heads -> 1 key head + 1 value head
```

This is a huge cache reduction. Compared with 32-head MHA, MQA stores 32x fewer K/V head vectors.

The tradeoff is that all query heads must use the same K/V representation. This can hurt model quality in real trained models, but it is very attractive for fast decode.


In [ ]:
for batch in BATCH_SIZES:
    for context in CONTEXTS:
        context_tokens, next_tokens = sample_context_batch(token_stream, batch=batch, context=context, seed=42)
        try:
            builder = TokenKVBuilder(VOCAB_SIZE, Q_HEADS, 1, HEAD_DIM).to(device=device, dtype=dtype).eval()
            with torch.inference_mode():
                q, k, v = builder.build(context_tokens, next_tokens)
            row = bench_grouped_decode_from_cache("MQA", q, k, v, WARMUP, ITERS)
            rows.append(row)
            print(f"MQA batch={batch:2d} context={context:5d} cache={row.cache_mb:8.1f} MB tok/s={row.tokens_per_sec:10.1f}")
            del builder, q, k, v
            torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"OOM: MQA batch={batch}, context={context}")


## 12. Let's Now Do GQA

**Grouped-Query Attention (GQA)** is the compromise between MHA and MQA.

Instead of giving every query head its own K/V pair, and instead of forcing all heads to share one K/V pair, GQA creates groups:

```text
32 query heads -> 8 key heads + 8 value heads
```

Each K/V head serves a group of query heads. In this notebook, GQA uses 8 KV heads, so the cache is 4x smaller than MHA.

Many modern open models use GQA because it preserves more head diversity than MQA while still reducing the KV cache substantially.


In [ ]:
for batch in BATCH_SIZES:
    for context in CONTEXTS:
        context_tokens, next_tokens = sample_context_batch(token_stream, batch=batch, context=context, seed=42)
        try:
            builder = TokenKVBuilder(VOCAB_SIZE, Q_HEADS, GQA_KV_HEADS, HEAD_DIM).to(device=device, dtype=dtype).eval()
            with torch.inference_mode():
                q, k, v = builder.build(context_tokens, next_tokens)
            row = bench_grouped_decode_from_cache(f"GQA ({GQA_KV_HEADS} KV heads)", q, k, v, WARMUP, ITERS)
            rows.append(row)
            print(f"GQA batch={batch:2d} context={context:5d} cache={row.cache_mb:8.1f} MB tok/s={row.tokens_per_sec:10.1f}")
            del builder, q, k, v
            torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"OOM: GQA batch={batch}, context={context}")


## 13. Let's Now Do MLA-style Latent Caching

**Multi-Head Latent Attention (MLA)** is different from MQA and GQA.

MQA and GQA reduce the number of physical KV heads. MLA-style caching instead stores a compressed latent vector per token:

```text
previous token -> compressed latent vector
```

At inference time, the model uses this compressed representation to recover the information needed for attention.

This notebook uses an **MLA-style latent cache** to demonstrate the memory idea. It is not a full DeepSeek MLA implementation. Full MLA includes additional projection absorption and kernel fusion tricks. Here, the educational point is simpler:

> Instead of caching full per-head K/V vectors, cache a smaller latent representation.


In [ ]:
for batch in BATCH_SIZES:
    for context in CONTEXTS:
        context_tokens, next_tokens = sample_context_batch(token_stream, batch=batch, context=context, seed=42)
        try:
            builder = TokenLatentBuilder(VOCAB_SIZE, Q_HEADS, MLA_LATENT_DIM).to(device=device, dtype=dtype).eval()
            with torch.inference_mode():
                q, latent = builder.build(context_tokens, next_tokens)
            row = bench_mla_latent_from_cache(q, latent, WARMUP, ITERS)
            rows.append(row)
            print(f"MLA batch={batch:2d} context={context:5d} cache={row.cache_mb:8.1f} MB tok/s={row.tokens_per_sec:10.1f}")
            del builder, q, latent
            torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"OOM: MLA-style batch={batch}, context={context}")


## 14. Let's Now Compare

Now we put the four methods side by side.

The expected cache-size pattern is:

```text
MHA: largest cache
GQA: 4x smaller than MHA
MLA-style: 8x smaller than MHA in this configuration
MQA: 32x smaller than MHA
```

Speed will not always scale exactly with cache size because kernels, softmax, launch overhead, and GPU utilization also matter. But at long context and reasonable batch size, the memory trend should become visible.


In [ ]:
def add_relative_columns(rows):
    by_shape = {}
    for row in rows:
        if row.method == "MHA":
            by_shape[(row.batch, row.context)] = row
    for row in rows:
        base = by_shape.get((row.batch, row.context))
        if base is None:
            continue
        row.speedup_vs_mha = row.tokens_per_sec / base.tokens_per_sec
        row.cache_reduction_vs_mha = base.cache_mb / row.cache_mb


add_relative_columns(rows)
df = pd.DataFrame([r.__dict__ for r in rows])
method_order = ["MHA", f"GQA ({GQA_KV_HEADS} KV heads)", "MQA", "MLA-style latent"]
df["method"] = pd.Categorical(df["method"], categories=method_order, ordered=True)
df = df.sort_values(["batch", "context", "method"]).reset_index(drop=True)
df.to_csv("tinystories_kv_decode_benchmark.csv", index=False)
display(df)


## 15. The Most Useful View: Batch 16

Batch 1 is noisy because fixed GPU overheads are large. Batch 16 with long contexts better shows the serving regime where the KV cache becomes a large memory object.


In [ ]:
summary_cols = [
    "method", "batch", "context", "cache_mb", "bytes_per_token_kb",
    "arithmetic_intensity", "ms_per_step", "tokens_per_sec",
    "speedup_vs_mha", "cache_reduction_vs_mha",
]

summary = df[(df["batch"] == 16) & (df["context"].isin([4096, 8192, 16384, 32768]))][summary_cols]
display(summary)


## 16. Plots

These plots show the whole story visually:

1. KV cache footprint
2. decode throughput
3. speedup compared with MHA


In [ ]:
plot_batch = max(df["batch"])
plot_df = df[df["batch"] == plot_batch].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for method, sub in plot_df.groupby("method", observed=True):
    sub = sub.sort_values("context")
    axes[0].plot(sub["context"], sub["cache_mb"], marker="o", linewidth=2, label=method)
    axes[1].plot(sub["context"], sub["tokens_per_sec"], marker="o", linewidth=2, label=method)
    axes[2].plot(sub["context"], sub["speedup_vs_mha"], marker="o", linewidth=2, label=method)

axes[0].set_title("KV cache footprint", fontweight="bold")
axes[0].set_ylabel("Cache MB")
axes[1].set_title("Decode throughput", fontweight="bold")
axes[1].set_ylabel("tokens/sec")
axes[2].set_title("Speedup vs MHA", fontweight="bold")
axes[2].set_ylabel("x")

for ax in axes:
    ax.set_xlabel("TinyStories context length")
    ax.set_xscale("log", base=2)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("tinystories_kv_decode_benchmark.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for method, sub in plot_df.groupby("method", observed=True):
    sub = sub.sort_values("context")
    axes[0].plot(sub["context"], sub["bytes_per_token_kb"], marker="o", linewidth=2, label=method)
    axes[1].plot(sub["context"], sub["arithmetic_intensity"], marker="o", linewidth=2, label=method)

axes[0].set_title("Bytes read per generated token", fontweight="bold")
axes[0].set_ylabel("KB/token")
axes[1].set_title("Arithmetic intensity", fontweight="bold")
axes[1].set_ylabel("FLOPs / byte of cache")

for ax in axes:
    ax.set_xlabel("TinyStories context length")
    ax.set_xscale("log", base=2)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("tinystories_kv_decode_memory_intensity.png", dpi=160, bbox_inches="tight")
plt.show()


## 17. What You Should Conclude

The core lesson is:

```text
MHA stores the most cache.
GQA stores less cache by sharing K/V within groups.
MQA stores the least cache by sharing one K/V pair across all query heads.
MLA-style caching stores a compressed latent representation instead of full K/V heads.
```

The experiment should show three patterns:

1. **Cache size grows linearly with context length.** Doubling context roughly doubles cache memory.
2. **Cache size grows linearly with batch size.** Batch 16 has 16x the cache of batch 1 for the same method/context.
3. **The attention variant changes the slope.** MHA grows fastest, GQA grows slower, MQA grows slowest, and MLA-style sits between them in this configuration.

Throughput will not always match cache reduction perfectly. GPU kernels, softmax cost, launch overhead, and tensor shapes all matter. But the long-context batch-16 rows should make the inference point clear: reducing physical cache size can strongly improve decode throughput.

Important caveat: this notebook is not measuring TinyStories language quality or perplexity. The cache builders are randomly initialized. This is an inference memory benchmark anchored to TinyStories contexts, not a trained model comparison.
